# Download Modules

In [ ]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu118
!pip install transformers sentencepiece datasets evaluate optuna rouge_score sacrebleu trl==0.8.0

In [ ]:
from datasets import load_dataset
from google.colab import drive
from IPython.display import display
from IPython.html import widgets
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from torch import optim
from torch.optim import AdamW
from torch.nn import functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm_notebook

sns.set()

In [ ]:
drive.mount('/content/gdrive')

In [ ]:
# Use 'google/mt5-small' for non-pro cloab users
model_repo = 'google/mt5-small'
model_path = '/content/gdrive/MyDrive/mt5_translation.pt'
max_seq_len = 20

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_repo)

In [ ]:
# Model description: https://huggingface.co/google/mt5-small
model = AutoModelForSeq2SeqLM.from_pretrained(model_repo)
model = model.cuda()

In [ ]:
token_ids = tokenizer.encode(
    '<id> This will be translated to Indonesian! (hopefully)',
    return_tensors='pt').cuda()
print(token_ids)

model_out = model.generate(token_ids)
print(model_out)

output_text = tokenizer.convert_tokens_to_string(
    tokenizer.convert_ids_to_tokens(model_out[0]))
print(output_text)

# Test Tokenizer

In [ ]:
example_input_str = '<id> This is just a test nbuig.'
input_ids = tokenizer.encode(example_input_str, return_tensors='pt')
print('Input IDs:', input_ids)

tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
print('Tokens:', tokens)

In [ ]:
sorted(tokenizer.vocab.items(), key=lambda x: x[1])

In [ ]:
# Source: https://huggingface.co/datasets/alt
dataset = load_dataset('alt')

In [ ]:
train_dataset = dataset['train']
test_dataset = dataset['test']

In [ ]:
train_dataset[0]

In [ ]:
LANG_TOKEN_MAPPING = {
    'en': '<en>',
    'id': '<id>'
}

In [ ]:
special_tokens_dict = {'additional_special_tokens': list(LANG_TOKEN_MAPPING.values())}
tokenizer.add_special_tokens(special_tokens_dict)
model.resize_token_embeddings(len(tokenizer))

In [ ]:
token_ids = tokenizer.encode(
    example_input_str, return_tensors='pt', padding='max_length',
    truncation=True, max_length=max_seq_len)
print(token_ids)

tokens = tokenizer.convert_ids_to_tokens(token_ids[0])
print(tokens)

In [ ]:
def encode_input_str(text, target_lang, tokenizer, seq_len,
                     lang_token_map=LANG_TOKEN_MAPPING):
  target_lang_token = lang_token_map[target_lang]

  # Tokenize and add special tokens
  input_ids = tokenizer.encode(
      text = target_lang_token + text,
      return_tensors = 'pt',
      padding = 'max_length',
      truncation = True,
      max_length = seq_len)

  return input_ids[0]

def encode_target_str(text, tokenizer, seq_len,
                      lang_token_map=LANG_TOKEN_MAPPING):
  token_ids = tokenizer.encode(
      text = text,
      return_tensors = 'pt',
      padding = 'max_length',
      truncation = True,
      max_length = seq_len)

  return token_ids[0]

def format_translation_data(translations, lang_token_map,
                            tokenizer, seq_len=128):
  input_lang, target_lang = 'en', 'id'

  # Get the translations for the batch
  input_text = translations[input_lang]
  target_text = translations[target_lang]

  if input_text is None or target_text is None:
    return None

  input_token_ids = encode_input_str(
      input_text, target_lang, tokenizer, seq_len, lang_token_map)

  target_token_ids = encode_target_str(
      target_text, tokenizer, seq_len, lang_token_map)

  return input_token_ids, target_token_ids

def transform_batch(batch, lang_token_map, tokenizer):
  inputs = []
  targets = []
  for translation_set in batch['translation']:
    formatted_data = format_translation_data(
        translation_set, lang_token_map, tokenizer, max_seq_len)

    if formatted_data is None:
      continue

    input_ids, target_ids = formatted_data
    inputs.append(input_ids.unsqueeze(0))
    targets.append(target_ids.unsqueeze(0))

  batch_input_ids = torch.cat(inputs).cuda()
  batch_target_ids = torch.cat(targets).cuda()

  return batch_input_ids, batch_target_ids

def get_data_generator(dataset, lang_token_map, tokenizer, batch_size=32):
  dataset = dataset.shuffle()
  for i in range(0, len(dataset), batch_size):
    raw_batch = dataset[i:i+batch_size]
    yield transform_batch(raw_batch, lang_token_map, tokenizer)

In [ ]:
# Testing `data_transform`
in_ids, out_ids = format_translation_data(
    train_dataset[0]['translation'], LANG_TOKEN_MAPPING, tokenizer)

print(' '.join(tokenizer.convert_ids_to_tokens(in_ids)))
print(' '.join(tokenizer.convert_ids_to_tokens(out_ids)))

# Testing data generator
data_gen = get_data_generator(train_dataset, LANG_TOKEN_MAPPING, tokenizer, 8)
data_batch = next(data_gen)
print('Input shape:', data_batch[0].shape)
print('Output shape:', data_batch[1].shape)

Hyperparameters

In [ ]:
# Constants
n_epochs = 4
batch_size = 16
print_freq = 50
checkpoint_freq = 1000
lr = 5e-4
n_batches = int(np.ceil(len(train_dataset) / batch_size))
total_steps = n_epochs * n_batches
n_warmup_steps = int(total_steps * 0.01)

Testing MT5-small to compare with supervised-finetuned model.

In [ ]:
import evaluate
bleu_metric = evaluate.load('sacrebleu')
rouge_metric = evaluate.load('rouge')
meteor_metric = evaluate.load('meteor')

# Get test data
model.eval()
test_generator = get_data_generator(test_dataset, LANG_TOKEN_MAPPING, tokenizer, batch_size)
all_preds = []
all_refs = []

# Collect predictions and references
for input_batch, label_batch in test_generator:
    with torch.no_grad():
        model_out = model.generate(input_ids=input_batch)
    preds = tokenizer.batch_decode(model_out, skip_special_tokens=True)
    refs = tokenizer.batch_decode(label_batch, skip_special_tokens=True)
    all_preds.extend(preds)
    all_refs.extend(refs)

# Store per-example scores
bleu_scores = []
rouge_scores = []
meteor_scores = []

for pred, ref in zip(all_preds, all_refs):
    # BLEU expects a list of references (even if it's just one)
    bleu = bleu_metric.compute(predictions=[pred], references=[[ref]])['score']
    rouge = rouge_metric.compute(predictions=[pred], references=[ref], rouge_types=["rougeL"], use_stemmer=True)['rougeL'] * 100
    meteor = meteor_metric.compute(predictions=[pred], references=[ref])['meteor'] * 100

    bleu_scores.append(bleu)
    rouge_scores.append(rouge)
    meteor_scores.append(meteor)

# Compute mean and standard deviation
print(f"BLEU Score: Mean = {np.mean(bleu_scores):.2f}, Std = {np.std(bleu_scores):.2f}")
print(f"ROUGE-L Score: Mean = {np.mean(rouge_scores):.2f}, Std = {np.std(rouge_scores):.2f}")
print(f"METEOR Score: Mean = {np.mean(meteor_scores):.2f}, Std = {np.std(meteor_scores):.2f}")


In [ ]:
# Optimizer
optimizer = AdamW(model.parameters(), lr=lr)
scheduler = get_linear_schedule_with_warmup(
    optimizer, n_warmup_steps, total_steps)

In [ ]:
losses = []

In [ ]:
def eval_model(model, gdataset, max_iters=8):
  test_generator = get_data_generator(gdataset, LANG_TOKEN_MAPPING,
                                      tokenizer, batch_size)
  eval_losses = []
  for i, (input_batch, label_batch) in enumerate(test_generator):
    if i >= max_iters:
      break

    model_out = model.forward(
        input_ids = input_batch,
        labels = label_batch)
    eval_losses.append(model_out.loss.item())

  return np.mean(eval_losses)

In [ ]:
for epoch_idx in range(n_epochs):
  # Randomize data order
  data_generator = get_data_generator(train_dataset, LANG_TOKEN_MAPPING,
                                      tokenizer, batch_size)

  for batch_idx, (input_batch, label_batch) \
      in tqdm_notebook(enumerate(data_generator), total=n_batches):
    optimizer.zero_grad()

    # Forward pass
    model_out = model.forward(
        input_ids = input_batch,
        labels = label_batch)

    # Calculate loss and update weights
    loss = model_out.loss
    losses.append(loss.item())
    loss.backward()
    optimizer.step()
    scheduler.step()

    # Print training update info
    if (batch_idx + 1) % print_freq == 0:
      avg_loss = np.mean(losses[-print_freq:])
      print('Epoch: {} | Step: {} | Avg. loss: {:.3f} | lr: {}'.format(
          epoch_idx+1, batch_idx+1, avg_loss, scheduler.get_last_lr()[0]))

    if (batch_idx + 1) % checkpoint_freq == 0:
      test_loss = eval_model(model, test_dataset)
      print('Saving model with test loss of {:.3f}'.format(test_loss))
      torch.save(model.state_dict(), model_path)

torch.save(model.state_dict(), model_path)

In [ ]:
torch.save(model.state_dict(), model_path)

In [ ]:
# Graph the loss

window_size = 50
smoothed_losses = []
for i in range(len(losses)-window_size):
  smoothed_losses.append(np.mean(losses[i:i+window_size]))

plt.plot(smoothed_losses[100:])

In [ ]:
test_sentence = test_dataset[0]['translation']['en']
print('Raw input text:', test_sentence)

input_ids = encode_input_str(
    text = test_sentence,
    target_lang = 'id',
    tokenizer = tokenizer,
    seq_len = model.config.max_length,
    lang_token_map = LANG_TOKEN_MAPPING)
input_ids = input_ids.unsqueeze(0).cuda()

print('Truncated input text:', tokenizer.convert_tokens_to_string(
    tokenizer.convert_ids_to_tokens(input_ids[0])))

In [ ]:
output_tokens = model.generate(input_ids, num_beams=10, num_return_sequences=3)
# print(output_tokens)
for token_set in output_tokens:
  print(tokenizer.decode(token_set, skip_special_tokens=True))

In [ ]:
# Get test data
model.eval()
test_generator = get_data_generator(test_dataset, LANG_TOKEN_MAPPING, tokenizer, batch_size)
all_preds = []
all_refs = []

# Collect predictions and references
for input_batch, label_batch in test_generator:
    with torch.no_grad():
        model_out = model.generate(input_ids=input_batch)
    preds = tokenizer.batch_decode(model_out, skip_special_tokens=True)
    refs = tokenizer.batch_decode(label_batch, skip_special_tokens=True)
    all_preds.extend(preds)
    all_refs.extend(refs)

# Store per-example scores
bleu_scores = []
rouge_scores = []
meteor_scores = []

for pred, ref in zip(all_preds, all_refs):
    # BLEU expects a list of references (even if it's just one)
    bleu = bleu_metric.compute(predictions=[pred], references=[[ref]])['score']
    rouge = rouge_metric.compute(predictions=[pred], references=[ref], rouge_types=["rougeL"], use_stemmer=True)['rougeL'] * 100
    meteor = meteor_metric.compute(predictions=[pred], references=[ref])['meteor'] * 100

    bleu_scores.append(bleu)
    rouge_scores.append(rouge)
    meteor_scores.append(meteor)

# Compute mean and standard deviation
print(f"BLEU Score: Mean = {np.mean(bleu_scores):.2f}, Std = {np.std(bleu_scores):.2f}")
print(f"ROUGE-L Score: Mean = {np.mean(rouge_scores):.2f}, Std = {np.std(rouge_scores):.2f}")
print(f"METEOR Score: Mean = {np.mean(meteor_scores):.2f}, Std = {np.std(meteor_scores):.2f}")
